# DisasterM3 — Architecture Ablation Study (Member B)
**Owner:** Abrar (covering for Tamanna)  
**Purpose:** Systematically ablate segmentation architectures (U-Net vs. U-Net++ vs. U-Net++ scSE vs. DeepLabV3+) holding data, loss, and training schedule constant.  
**Hardware:** Kaggle T4 GPU (16 GB VRAM) — PyTorch AMP (FP16).

### Assigned Experiments
| ID | Architecture | SMP Call | Key Feature |
|---|---|---|---|
| **B1** | Baseline U-Net | `smp.Unet(encoder_name='resnet34')` | Baseline (0.4619 mIoU) |
| **B2** | U-Net++ | `smp.UnetPlusPlus(encoder_name='resnet34')` | Dense nested skip pathways |
| **B3** | U-Net++ (scSE) | `smp.UnetPlusPlus(..., decoder_attention_type='scse')` | Spatial + Channel Squeeze-Excitation |
| **B4** | DeepLabV3+ | `smp.DeepLabV3Plus(encoder_name='resnet34')` | Atrous Spatial Pyramid Pooling (ASPP) |
| **B5** | MA-Net (Stretch) | `smp.MAnet(encoder_name='resnet34')` | Multi-scale Attention for rare/small regions |

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print('⏳ Installing segmentation-models-pytorch...')
    !pip install -q segmentation-models-pytorch==0.3.4
    import segmentation_models_pytorch as smp

print(f'✓ Environment ready. smp version: {smp.__version__}')

⏳ Installing segmentation-models-pytorch...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.0 MB/s eta 0:00:00
✓ Environment ready. smp version: 0.3.4


In [2]:
# ── Cell 2: Configuration & Architecture Selector ────────────────────────
import os, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path


# ── Data Root ──
DATA_ROOT = Path('/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct')
MANIFEST_PATH = DATA_ROOT / 'train_release.json'

# ── ARCHITECTURE SELECTOR (Select experiment to run) ──
# Choices: 'B1_Unet', 'B2_UnetPlusPlus', 'B3_UnetPlusPlus_scse', 'B4_DeepLabV3Plus', 'B5_MAnet'
EXPERIMENT_ID = 'B3_UnetPlusPlus_scse'

# Backbone held constant across all architecture tests
ENCODER_NAME = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
NUM_CLASSES = 4  # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed

# ── Training config ──
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40
BATCH_SIZE = 8  # If DeepLabV3+ OOMs, reduce to 8
NUM_WORKERS = 2
IMAGE_SIZE = 512

# ── Checkpointing ──
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
TIME_LIMIT_HOURS = 11.4

# ── Output ──
OUTPUT_DIR = '/kaggle/working/ablation_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'✓ Configuration loaded:')
print(f'  Experiment: {EXPERIMENT_ID}')
print(f'  Encoder:    {ENCODER_NAME} (pretrained={ENCODER_WEIGHTS})')
print(f'  Batch Size: {BATCH_SIZE} | Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'  Device:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'  VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

✓ Configuration loaded:
  Experiment: B3_UnetPlusPlus_scse
  Encoder:    resnet34 (pretrained=imagenet)
  Batch Size: 8 | Image Size: 512x512
  Device:     Tesla T4
  VRAM:       15.6 GB


In [3]:
# ── Cell 4: Build combined multi-class masks ──────────────────────────────
import cv2, numpy as np, os, json
from collections import defaultdict
from pathlib import Path

with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

bda_entries = [
    e for e in raw_data
    if e.get('task') == 'Referring Expression Segmentation'
    and e.get('cls_description') == 'Building Damage Assessment'
]

def resolve_path(rel_path):
    if not rel_path: return None
    rel_path = rel_path.replace('\\', '/')
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / 'train_images' / rel_path,
        DATA_ROOT / 'train_images' / 'train_images' / filename,
        DATA_ROOT / 'masks' / rel_path,
        DATA_ROOT / 'masks' / 'masks' / filename,
    ]
    for c in candidates:
        if c.exists(): return str(c)
    return None

by_image = defaultdict(dict)
for e in bda_entries:
    post_img = e.get('post_image_path', '')
    if e.get('image_type') != 'Optical': continue
    mask_rel = e.get('ground_truth', '')
    folder = Path(mask_rel.replace('\\', '/')).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved

FOLDER_TO_CLASS = {
    'train_building_intact_mask': 1,
    'train_building_damaged_mask': 2,
    'train_building_destroyed_mask': 3,
}

os.makedirs('/tmp/combined_masks', exist_ok=True)
pairs = []
for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path: continue
    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if not mask_path: continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id
    if combined_mask is None: continue
    mask_filename = Path(img_path).name
    save_path = f'/tmp/combined_masks/{mask_filename}'
    cv2.imwrite(save_path, combined_mask)
    pairs.append({'image_path': img_path, 'mask_path': save_path})

print(f'✓ Built {len(pairs):,} valid (image, mask) pairs.')

✓ Built 6,443 valid (image, mask) pairs.


In [4]:
# ── Cell 5: Datasets, Augmentation & DataLoaders ──────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class DisasterM3SegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        image = cv2.imread(pair['image_path'])
        if image is None: return self.__getitem__((idx + 1) % len(self))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(pair['mask_path'], cv2.IMREAD_GRAYSCALE)
        if mask is None: return self.__getitem__((idx + 1) % len(self))
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        return image, mask.long()

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
train_dataset = DisasterM3SegDataset(train_pairs, transform=train_transform)
val_dataset = DisasterM3SegDataset(val_pairs, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'✓ Train samples: {len(train_dataset):,} ({len(train_loader):,} batches)')
print(f'✓ Val samples:   {len(val_dataset):,} ({len(val_loader):,} batches)')

✓ Train samples: 5,798 (725 batches)
✓ Val samples:   645 (81 batches)


In [5]:
# ── Cell 6: Loss Function (Standard Baseline CE + Dice for strict ablation) ─
import torch.nn as nn
import torch.nn.functional as F

class_pixel_counts = torch.tensor([6267276585, 395903867, 70675310, 22119406], dtype=torch.float)
total_pixels = class_pixel_counts.sum()
class_weights = total_pixels / (NUM_CLASSES * class_pixel_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_per_class.mean()

class CombinedLoss(nn.Module):
    def __init__(self, weights, num_classes):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights)
        self.dice = DiceLoss(num_classes)
    def forward(self, logits, targets):
        l_ce = self.ce(logits, targets)
        l_dice = self.dice(logits, targets)
        return 0.5 * l_ce + 0.5 * l_dice, l_ce.item(), l_dice.item()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = CombinedLoss(class_weights.to(DEVICE), NUM_CLASSES)
print(f'✓ Baseline CE+Dice Loss initialized on {DEVICE}')

✓ Baseline CE+Dice Loss initialized on cuda


In [6]:
# ── Cell 8: Model Instantiation via Architecture Selector ────────────────
import segmentation_models_pytorch as smp

print(f'Building Architecture for {EXPERIMENT_ID}...')

if EXPERIMENT_ID == 'B1_Unet':
    model = smp.Unet(
        encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=NUM_CLASSES
    )
elif EXPERIMENT_ID == 'B2_UnetPlusPlus':
    model = smp.UnetPlusPlus(
        encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=NUM_CLASSES
    )
elif EXPERIMENT_ID == 'B3_UnetPlusPlus_scse':
    model = smp.UnetPlusPlus(
        encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=NUM_CLASSES,
        decoder_attention_type='scse'
    )
elif EXPERIMENT_ID == 'B4_DeepLabV3Plus':
    model = smp.DeepLabV3Plus(
        encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=NUM_CLASSES
    )
elif EXPERIMENT_ID == 'B5_MAnet':
    model = smp.MAnet(
        encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=NUM_CLASSES
    )
else:
    raise ValueError(f'Unknown EXPERIMENT_ID: {EXPERIMENT_ID}')

model = model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'✓ Model successfully loaded: {EXPERIMENT_ID}')
print(f'  Total Parameters: {total_params:,}')

Building Architecture for B3_UnetPlusPlus_scse...
Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /root/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 299MB/s]


✓ Model successfully loaded: B3_UnetPlusPlus_scse
  Total Parameters: 26,281,767


In [7]:
# ── Cell 9: Training Loop with AMP and ModelCheckpoint ───────────────────
import time, datetime
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = GradScaler()

def compute_iou(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        p_cls = (pred == cls)
        t_cls = (target == cls)
        intersection = (p_cls & t_cls).sum().item()
        union = (p_cls | t_cls).sum().item()
        ious.append(float('nan') if union == 0 else intersection / union)
    return ious

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    for images, masks in tqdm(loader, desc='Train', leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(images)
            loss, _, _ = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_ious = [[] for _ in range(NUM_CLASSES)]
    for images, masks in tqdm(loader, desc='Val', leave=False):
        images, masks = images.to(device), masks.to(device)
        with autocast():
            logits = model(images)
            loss, _, _ = criterion(logits, masks)
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        for pred, mask in zip(preds, masks):
            ious = compute_iou(pred.cpu(), mask.cpu(), NUM_CLASSES)
            for cls, val in enumerate(ious):
                if not np.isnan(val):
                    all_ious[cls].append(val)
    class_ious = [np.mean(c) if c else 0.0 for c in all_ious]
    miou = np.mean([iou for iou in class_ious if iou > 0])
    return total_loss / len(loader), miou

best_miou = 0.0
epoch_times = []
history = {'train_loss': [], 'val_loss': [], 'miou': []}
best_model_path = f'/kaggle/working/best_model_{EXPERIMENT_ID}.pth'

print(f'Starting 40-Epoch Training for {EXPERIMENT_ID}...')
for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, miou = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    dt = time.time() - t0
    epoch_times.append(dt)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['miou'].append(miou)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {miou:.4f} | Time: {dt:.1f}s')
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), best_model_path)
        print(f'  ✓ Saved new best model: mIoU={best_miou:.4f}')

print(f'\n✓ Training finished for {EXPERIMENT_ID}. Best mIoU: {best_miou:.4f}')

/tmp/ipykernel_23/1265608967.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Starting 40-Epoch Training for B3_UnetPlusPlus_scse...


Train:   0%|          | 0/725 [00:00<?, ?it/s]/tmp/ipykernel_23/1265608967.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/81 [00:00<?, ?it/s]/tmp/ipykernel_23/1265608967.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/40 | Train Loss: 0.8088 | Val Loss: 0.7232 | Val mIoU: 0.2741 | Time: 723.4s
  ✓ Saved new best model: mIoU=0.2741


Epoch 02/40 | Train Loss: 0.7235 | Val Loss: 0.7338 | Val mIoU: 0.2646 | Time: 723.4s


...

Epoch 36/40 | Train Loss: 0.4364 | Val Loss: 0.5537 | Val mIoU: 0.3550 | Time: 723.0s


Epoch 37/40 | Train Loss: 0.4312 | Val Loss: 0.5529 | Val mIoU: 0.3590 | Time: 723.1s


Epoch 38/40 | Train Loss: 0.4330 | Val Loss: 0.5627 | Val mIoU: 0.3584 | Time: 723.6s


Epoch 39/40 | Train Loss: 0.4334 | Val Loss: 0.5487 | Val mIoU: 0.3565 | Time: 722.6s


Epoch 40/40 | Train Loss: 0.4299 | Val Loss: 0.5535 | Val mIoU: 0.3607 | Time: 723.2s

✓ Training finished for B3_UnetPlusPlus_scse. Best mIoU: 0.3627


In [8]:
# ── Cell 10: Final Evaluation & Structured CSV Export ────────────────────
import csv

@torch.no_grad()
def evaluate_final(model, loader, device):
    model.eval()
    intersection = torch.zeros(NUM_CLASSES)
    union = torch.zeros(NUM_CLASSES)
    tp = torch.zeros(NUM_CLASSES)
    fp = torch.zeros(NUM_CLASSES)
    fn = torch.zeros(NUM_CLASSES)
    for images, masks in tqdm(loader, desc='Final Eval'):
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1)
        for c in range(NUM_CLASSES):
            pred_c = (preds == c)
            true_c = (masks == c)
            intersection[c] += (pred_c & true_c).sum().item()
            union[c] += (pred_c | true_c).sum().item()
            tp[c] += (pred_c & true_c).sum().item()
            fp[c] += (pred_c & ~true_c).sum().item()
            fn[c] += (~pred_c & true_c).sum().item()
    iou = intersection / (union + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return iou, precision, recall, f1

# Load best weights
model.load_state_dict(torch.load(best_model_path))
iou, prec, rec, f1 = evaluate_final(model, val_loader, DEVICE)

class_names = ['Background', 'Intact', 'Damaged', 'Destroyed']
print(f'\nFinal Benchmark for {EXPERIMENT_ID}:')
print(f'{"Class":<12} {"IoU":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
for c in range(NUM_CLASSES):
    print(f'{class_names[c]:<12} {iou[c]:>8.4f} {prec[c]:>10.4f} {rec[c]:>8.4f} {f1[c]:>8.4f}')

miou_val = float(iou.mean())
print(f'\nOverall Mean IoU: {miou_val:.4f}')
print(f'Overall Mean F1:  {float(f1.mean()):.4f}')

# Export clean CSV without template errors!
csv_path = f'/kaggle/working/{EXPERIMENT_ID}.csv'
row = {
    'experiment_id': EXPERIMENT_ID,
    'variable': 'architecture',
    'value': EXPERIMENT_ID,
    'miou': miou_val,
    'bg_iou': float(iou[0]),
    'intact_iou': float(iou[1]),
    'damaged_iou': float(iou[2]),
    'destroyed_iou': float(iou[3]),
    'precision': float(prec.mean()),
    'recall': float(rec.mean()),
    'f1': float(f1.mean()),
    'train_time_hrs': sum(epoch_times) / 3600,
    'notes': f'ResNet-34 backbone, CE+Dice, 40 epochs, batch 16'
}
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    w.writeheader()
    w.writerow(row)

print(f'✓ Exported structured CSV to: {csv_path}')

Final Eval: 100%|██████████| 81/81 [00:35<00:00,  2.31it/s]


Final Benchmark for B3_UnetPlusPlus_scse:
Class             IoU  Precision   Recall       F1
Background     0.9135     0.9948   0.9179   0.9548
Intact         0.4772     0.5116   0.8765   0.6461
Damaged        0.2311     0.2571   0.6960   0.3755
Destroyed      0.1252     0.1330   0.6791   0.2225

Overall Mean IoU: 0.4367
Overall Mean F1:  0.5497
✓ Exported structured CSV to: /kaggle/working/B3_UnetPlusPlus_scse.csv
